# Multi-label Tweet Topic Classification

This notebook demonstrates a multi-label tweet topic classification task using a dataset from Hugging Face. The goal is to preprocess tweet data, build a deep learning model (LSTM-based), train it, and evaluate its performance on classifying tweets into various topics.

In [9]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("cardiffnlp/tweet_topic_multi")

In [10]:
ds

DatasetDict({
    test_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 573
    })
    test_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1679
    })
    train_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 4585
    })
    train_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1505
    })
    train_all: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 6090
    })
    validation_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 573
    })
    validation_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 188
    })
    train_random: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 4564
    })
    validati

In [11]:
type(ds)

datasets.dataset_dict.DatasetDict

In [12]:
import pandas as pd
train = pd.DataFrame(ds['train_2020'])
display(train.head())

test = pd.DataFrame(ds['test_2020'])
valid = pd.DataFrame(ds['validation_2020'])

,text,date,label,label_name,id
0,The {@Clinton LumberKings@} beat the {@Cedar R...,2019-09-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1170516324419866624
1,I would rather hear Eli Gold announce this Aub...,2019-09-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1170516440690176006
2,"Someone take my phone away, I’m trying to not ...",2019-09-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1170516543387709440
3,"A year ago, Louisville struggled to beat an FC...",2019-09-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1170516620466429953
4,Anyone know why the #Dodgers #Orioles game nex...,2019-09-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1170516711411310592


In [13]:
train['label_name'][2][0]

'sports'

In [14]:
def remove_cols(df):
  return df.drop(columns=['date','id'])

def labels_array_to_string(df):
  for i in range(0,df.shape[0]):
    if len(df.loc[i,'label_name']) > 0:
      df.loc[i,'label_name'] = df['label_name'][i][0]
    else :
      df.loc[i,'label_name'] = ''
  return df


In [15]:
# !pip install urlextract

In [16]:
import re
from urlextract import URLExtract
extractor = URLExtract()

def format_tweet(tweet):
    # mask web urls
    urls = extractor.find_urls(tweet)
    for url in urls:
        tweet = tweet.replace(url, "{{URL}}")
    # format twitter account
    tweet = re.sub(r"\b(\s*)(@[\S]+)\b", r'\1{\2@}', tweet)
    return tweet

#Data Preprocessing

## Data Preprocessing Overview

This section handles the initial preparation of the raw tweet data. It involves loading the datasets into pandas DataFrames, dropping irrelevant columns like `date` and `id`, and transforming the `label_name` column from an array of labels to a single string representation. Additionally, the `format_tweet` function is applied to clean and standardize tweet text by masking URLs and formatting Twitter account mentions.

In [17]:
train = remove_cols(train)
test = remove_cols(test)
valid = remove_cols(valid)


In [18]:
train = labels_array_to_string(train)
test = labels_array_to_string(test)
valid = labels_array_to_string(valid)

In [19]:
train['text'] = train['text'].apply(format_tweet)
test['text'] = test['text'].apply(format_tweet)
valid['text'] = valid['text'].apply(format_tweet)

In [20]:
train = train[train['label_name'] != '' ]

In [21]:
num_classes = len(train['label_name'].unique())

In [22]:
train.isnull().sum()

,0
text,0
label,0
label_name,0


In [23]:
test.isnull().sum()

,0
text,0
label,0
label_name,0


In [24]:
valid.isnull().sum()

,0
text,0
label,0
label_name,0


## Tokenization

In this step, the `text` data from the tweets is tokenized. Tokenization involves breaking down the raw text into individual words or subwords (tokens). A custom tokenizer is used to extract relevant terms, converting them to lowercase and handling hashtags and mentions. The resulting tokens are stored in a new `tokens` column in the DataFrame.

In [25]:
import re
def tokenizer(text):
  return re.findall(r'[#@]?\w+', text.lower())

In [26]:
train['tokens'] = train['text'].apply(tokenizer)
test['tokens'] = test['text'].apply(tokenizer)
valid['tokens'] = valid['text'].apply(tokenizer)

## Vocabulary Mapping

This section creates a numerical representation for each unique word (token) in the dataset. A vocabulary map (`vocab`) is built where each unique token is assigned a unique integer ID. Special tokens like `<PAD>` and `<UNK>` (unknown) are also included. The `tokens` are then converted into numerical sequences (`sequence`) based on this vocabulary. This step also determines the total vocabulary size (`vocab_size`) and the maximum sequence length (`max_len`) observed in the training data.

In [27]:
# def vocab_map(df):
#   vocab = {
#       '<PAD>': 0,
#       '<UNK>': 1
#   }

#   idx = 2

#   for tokens in df["tokens"]:
#       for word in tokens:
#           if word not in vocab:
#               vocab[word] = idx
#               idx += 1

#   df["sequence"] = df["tokens"].apply(
#       lambda tokens: [vocab.get(word, 1) for word in tokens]
#   )
#   return df, len(vocab), len(df['tokens'].max())

# train, vocab_size, max_len= vocab_map(train)

In [28]:
# 1. Build vocab ONLY on training tokens
def build_vocab(df):
    vocab = {'<PAD>': 0, '<UNK>': 1}
    idx = 2
    for tokens in df["tokens"]:
        for word in tokens:
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab

train_vocab = build_vocab(train)

# 2. Encode all splits using the SAME vocab
def encode_sequences(df, vocab):
    df["sequence"] = df["tokens"].apply(
        lambda tokens: [vocab.get(word, 1) for word in tokens]
    )
    return df

train = encode_sequences(train, train_vocab)
test  = encode_sequences(test, train_vocab)
valid = encode_sequences(valid, train_vocab)

vocab_size = len(train_vocab)
max_len    = max(train["tokens"].apply(len))   # compute only from training

In [29]:
train.head()

,text,label,label_name,tokens,sequence
0,The {@Clinton LumberKings@} beat the {@Cedar R...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sports,"[the, @clinton, lumberkings, beat, the, @cedar...","[2, 3, 4, 5, 2, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,I would rather hear Eli Gold announce this Aub...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sports,"[i, would, rather, hear, eli, gold, announce, ...","[38, 39, 40, 41, 42, 43, 44, 45, 46, 12, 47, 4..."
2,"Someone take my phone away, I’m trying to not ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sports,"[someone, take, my, phone, away, i, m, trying,...","[51, 52, 53, 54, 55, 38, 56, 57, 58, 59, 60, 6..."
3,"A year ago, Louisville struggled to beat an FC...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sports,"[a, year, ago, louisville, struggled, to, beat...","[21, 67, 68, 69, 70, 58, 5, 71, 72, 73, 74, 75..."
4,Anyone know why the #Dodgers #Orioles game nex...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sports,"[anyone, know, why, the, #dodgers, #orioles, g...","[99, 100, 101, 2, 102, 103, 12, 104, 105, 106,..."


## Padding

To ensure that all input sequences to the neural network have a uniform length, padding is applied. Sequences shorter than `max_len` are padded with `0` (corresponding to `<PAD>` token), while sequences longer than `max_len` are truncated. This creates a `padded_sequence` column, which is essential for batch processing in deep learning models.

In [30]:
def padding(df):
    padded_dataset = []

    for sequence in df['sequence']:
        if len(sequence) < max_len:
            sequence = sequence + [0] * (max_len - len(sequence))
        else:
            sequence = sequence[:max_len]

        padded_dataset.append(sequence)

    df['padded_sequence'] = padded_dataset
    return df


In [31]:
train = padding(train)
test  = padding(test)
valid = padding(valid)

# Embedding with LSTM and Classifier


## Model Building

Here, a sequential deep learning model is constructed using Keras. The model architecture consists of:

1.  **Embedding Layer**: Converts integer-encoded words into dense vectors of fixed size.
2.  **LSTM Layer**: A Long Short-Term Memory recurrent layer, suitable for processing sequential data like text, with dropout for regularization.
3.  **Dense Layers**: Fully connected layers with ReLU activation for feature transformation.
4.  **Dropout Layer**: Further regularization to prevent overfitting.
5.  **Output Dense Layer**: A final dense layer with `sigmoid` activation, appropriate for multi-label classification, where each output neuron corresponds to a class and predicts the probability of that class being present.

The model is compiled with the `adam` optimizer and `binary_crossentropy` loss function, which is standard for multi-label tasks.

In [32]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

def build_model(vocab_size, max_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, 64, input_length=max_len),
        Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='sigmoid')   # multi-label
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',                # correct for multi-label
        metrics=['binary_accuracy']                # better than 'accuracy'
    )
    return model

In [33]:
model = build_model(vocab_size, max_len, num_classes)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
train['padded_sequence'].apply(len).unique()

array([63])

In [35]:
import numpy as np

X_train = np.array(train['padded_sequence'].tolist())

print(X_train.shape)
print(X_train.dtype)
type(train['label'][0])
y_train = np.array(train['label'].tolist())
print(y_train.shape)

(4584, 63)
int64
(4584, 19)


Forming testing and validation data

In [36]:
# Convert padded sequences to numpy arrays for test and validation
X_test = np.array(test['padded_sequence'].tolist())
X_valid = np.array(valid['padded_sequence'].tolist())

# Convert labels to numpy arrays for test and valid
y_test = np.array(test['label'].tolist())
y_valid = np.array(valid['label'].tolist())

print('Shape of X_test:', X_test.shape)
print('Shape of y_test:', y_test.shape)
print('Shape of X_valid:', X_valid.shape)
print('Shape of y_valid:', y_valid.shape)

Shape of X_test: (573, 63)
Shape of y_test: (573, 19)
Shape of X_valid: (573, 63)
Shape of y_valid: (573, 19)


## Training and Testing the Model

This section involves training the defined LSTM model and evaluating its performance. Class weights are computed to address potential class imbalance in the training data, ensuring that the model doesn't overemphasize majority classes. The model is then trained using the `X_train` and `y_train` data, with `X_valid` and `y_valid` used for validation during training. After training, the model's performance is assessed on the unseen `X_test` and `y_test` data, and a classification report provides detailed metrics such as precision, recall, and F1-score for each class.

In [37]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = {}
for i in range(num_classes):
    class_weights[i] = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1]),
        y=y_train[:, i]
    )[1]   # weight for class 1 (positive)

In [38]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=30,
    batch_size=64,
    class_weight=class_weights,   # compute per-class weights for multi-label
    callbacks=[early_stop]
)

Epoch 1/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 48s 387ms/step - binary_accuracy: 0.8113 - loss: 2.3111 - val_binary_accuracy: 0.9087 - val_loss: 0.2961
Epoch 2/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 27s 373ms/step - binary_accuracy: 0.9022 - loss: 1.6862 - val_binary_accuracy: 0.9087 - val_loss: 0.2824
Epoch 3/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 27s 373ms/step - binary_accuracy: 0.9076 - loss: 1.5896 - val_binary_accuracy: 0.9087 - val_loss: 0.2701
Epoch 4/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 28s 383ms/step - binary_accuracy: 0.9121 - loss: 1.4454 - val_binary_accuracy: 0.9194 - val_loss: 0.2412
Epoch 5/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 45s 441ms/step - binary_accuracy: 0.9221 - loss: 1.2916 - val_binary_accuracy: 0.9259 - val_loss: 0.2183
Epoch 6/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 27s 371ms/step - binary_accuracy: 0.9292 - loss: 1.1812 - val_binary_accuracy: 0.9310 - val_loss: 0.2139
Epoch 7/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 42s 380ms/step - binary_accuracy: 0.9331 - loss: 1.0875 - val_binary_accuracy: 0.9312 - val_loss: 0.2093

In [39]:
y_pred_proba = model.predict(X_test)
y_pred_binary = (y_pred_proba > 0.5).astype(int)

from sklearn.metrics import f1_score
print("Weighted F1:", f1_score(y_test, y_pred_binary, average='weighted'))

18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 121ms/step
Weighted F1: 0.4783248839184919


In [40]:
y_pred_proba = model.predict(X_test)
y_pred_binary = (y_pred_proba > 0.5).astype(int)
from sklearn.metrics import f1_score
print(f1_score(y_test, y_pred_binary, average='weighted'))

18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step
0.4783248839184919


In [44]:
from sklearn.metrics import classification_report

y_pred_proba = model.predict(X_test)
y_pred_binary = (y_pred_proba > 0.5).astype(int)

# Get class names
class_names = train['label_name'].unique()
print(classification_report(y_test, y_pred_binary, target_names=class_names))

# Also check which classes are NEVER predicted
print("\nClasses never predicted:")
for i, name in enumerate(class_names):
    if y_pred_binary[:, i].sum() == 0:
        print(f"  - {i},{name}")

18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 123ms/step
                          precision    recall  f1-score   support

                  sports       0.75      0.18      0.29        33
                  gaming       0.00      0.00      0.00        24
 celebrity_&_pop_culture       0.72      0.35      0.47        94
   news_&_social_concern       0.53      0.10      0.17        77
          arts_&_culture       0.50      0.25      0.33        16
           food_&_dining       1.00      0.17      0.29         6
         film_tv_&_video       0.70      0.21      0.32        90
        fitness_&_health       0.00      0.00      0.00        20
         fashion_&_style       0.00      0.00      0.00        10
    diaries_&_daily_life       0.00      0.00      0.00        20
business_&_entrepreneurs       0.00      0.00      0.00        15
    science_&_technology       0.94      0.66      0.78        95
                   music       0.89      0.64      0.74       195
  learning_&_educational       0.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [50]:
train['label_name'].value_counts()

,count
label_name,
news_&_social_concern,819
sports,756
celebrity_&_pop_culture,669
diaries_&_daily_life,555
film_tv_&_video,383
music,290
arts_&_culture,256
business_&_entrepreneurs,198
fitness_&_health,150


In [41]:
x = pd.DataFrame(ds['train_2020'])

In [42]:
train['label'].apply(sum).value_counts()

,count
label,
1,2210
2,1661
3,572
4,124
5,15
6,2


In [51]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 63, 64)         │     1,240,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 19)             │           627 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,934,907 (15.01 MB)

 Trainable params: 1,311,635 (5.00 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,623,272 (10.01 MB)

In [52]:
model.save('tweet_v1.keras')